AI 사용내역 : https://claude.ai/share/1d65e6b6-aa86-499e-9a13-0d3f6f9772f5

In [25]:
%pip install python-dotenv requests


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY: str = os.environ["KOREAN_DICT_KEY"]
print(KEY[:4] + "***") #키 전체가 출력되지 않도록 앞 4자만 확

B3AA***


Q1

(a)

(1) 코드

In [27]:
import requests

def search_word(q: str, num: int = 10, start: int = 1) -> dict:
    """우리말샘에서 q를 검색한 결과를 dict로 돌려준다."""
    params = {
        "key": KEY, "q": q, "req_type": "json",
        "num": num, "start": start, 
        "type1": "word",
    }

    r = requests.get(
        "https://opendict.korean.go.kr/api/search",
        params=params, timeout=10,
    )
    
    r.raise_for_status()
    return r.json()

(2) 함수를 정의하는 셀이므로 코드 실행 결과는 없다.

(3) 설명 : requests.get으로 우리말샘 엔드포인트를 호출하며 key, q, req_type="json", num, start, type1="word"를 쿼리 매개변수로 넘겼다. timeout=10을 명시하고 r.raise_for_status()로 HTTP 오류를 먼저 검사한 뒤, r.json()으로 JSON 응답을 파이썬 dict로 파싱해 반환했다.

(b)

(1) 코드

In [34]:
data = search_word("김치")

import json
print(json.dumps(data, ensure_ascii=False, indent=2)[:400])

{
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",
          


(2) 코드 실행 결과 : {
  "channel": {
    "total": 328,
    "num": 10,
    "title": "우리말샘 개발 지원(Open API) - 사전 어휘 검색",
    "start": 1,
    "description": "우리말샘 개발 지원(Open API) - 사전 어휘 검색 결과",
    "link": "https://opendict.korean.go.kr",
    "item": [
      {
        "word": "김치",
        "sense": [
          {
            "syntacticArgument": "",
            "syntacticAnnotation": "",
            "cat": "",

(3) 설명 : data = search_word("김치")를 우선 호출하고 문제에 나온 코드를 실행해 data의 구조를 400번째까지 출력했다. ensure_ascii=False를 빼면 기본값 True가 적용되어 한글이 유니코드로 이스케이프된다.

(c)

(1) 코드

In [ ]:
items: list[dict] = data["channel"]["item"]
total: int = data["channel"]["total"]
n: int = len(items)
print(f"총 {total}건, 이 페이지 {n}건")

for item in items[:5]:
    word: str = item["word"]
    pos: str = item["sense"][0].get("pos", "품사 없음") #sense가 리스트이므로 인덱스로 접근
    definition: str = item["sense"][0]["definition"] #sense가 리스트이므로 인덱스로 접근
    print(f"{word} ({pos}) -> {definition[:40]})")

총 328건, 이 페이지 10건
김치 (명사) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린)
김-치 (명사) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 )
김-치 (명사) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南)
김치 공장 () -> 김치를 만드는 공장.)
김치 보릿고개 () -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 )


(2) 총 328건, 이 페이지 10건
김치 (명사) -> 소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린)
김-치 (명사) -> 고려 말기·조선 초기의 문신(?~?). 자는 기보(基甫). 김해 부사를 )
김-치 (명사) -> 조선 중기의 문신(1577~1625). 자는 사정(士精). 호는 남봉(南)
김치 공장 () -> 김치를 만드는 공장.)
김치 보릿고개 () -> 김장철인 가을·겨울과 달리 상대적으로 김치가 부족한 봄여름을 비유적으로 )

(3) 설명 : data["channel"]["total"]로 전체 결과 수를, len(items)로 이 페이지에 받은 항목 수를 출력했다. 모든 항목에 pos가 있는 건 아니므로 dict.get()을 사용해 누락 시 "품사 없음"을 기본값으로 두었고, 뜻풀이는 sense 딕셔너리 안의 definition을 꺼내 앞 40자만 표시하였다. (b)코드 실행 결과를 보아 sense는 리스트이고, item["sense"][1]["definition"]으로 코드를 작성했을 때 indexError가 나서 sense 리스트에 요소가 하나뿐임을 파악하고 pos = item["sense"][0].get으로, definition = item["sense"][0]["definition"]으로 수정했다. 

Q2

(a)

(1) 코드

In [32]:
import time

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

results: dict[str, dict] = {}
for w in words:
    data = search_word(w)
    results[w] = data
    total = data["channel"]["total"]
    print(f"{w} : {total}건")
    time.sleep(0.3)

김치 : 328건
라면 : 86건
만두 : 89건
김밥 : 39건
국수 : 227건
떡볶이 : 24건
불고기 : 38건
비빔밥 : 38건


(2) 코드 실행 결과 : 김치 : 328건
라면 : 86건
만두 : 89건
김밥 : 39건
국수 : 227건
떡볶이 : 24건
불고기 : 38건
비빔밥 : 38건 

(3) 설명 : for문으로 한 검색어씩 search_word(w)를 호출하고, 매 호출 사이에 time.sleep(0.3)을 넣어 서버에 부담을 주지 않았다. 각 응답에서 data["channel"]["total"]을 꺼내 검색어별 전체 결과 수를 출력했다. 

(b)

힌트대로 item.get("pos")로 코드를 작성했더니 전부 다 미상으로 처리됐다. 이에 따라 pos가 어디에 존재하는지 알아보기 위해 아래와 같은 코드를 작성하여 한 항목의 구조를 출력해 확인했다. 그 결과 pos가 item 바로 아래가 아니라 sense 리스트의 0번 인덱스에 존재한다는 것을 파악하여 item.["sense"][0]으로 코드를 수정했다. Q1의 (c)에서 발생한 문제와 같은 것임을 파악했다.

In [ ]:
import json
print(json.dumps(all_items[0], ensure_ascii=False, indent=2))

{
  "word": "김치",
  "sense": [
    {
      "syntacticArgument": "",
      "syntacticAnnotation": "",
      "cat": "",
      "definition": "소금에 절인 배추나 무 따위를 고춧가루, 파, 마늘 따위의 양념에 버무린 뒤 발효를 시킨 음식. 재료와 조리 방법에 따라 많은 종류가 있다.",
      "link": "https://opendict.korean.go.kr/dictionary/view?sense_no=107717",
      "origin": "",
      "sense_no": "001",
      "target_code": "107717",
      "type": "",
      "pos": "명사"
    }
  ]
}


(1) 코드

In [ ]:
import time
from collections import Counter

words: list[str] = [
    "김치", "라면", "만두", "김밥",
    "국수", "떡볶이", "불고기", "비빔밥",
]

all_items: list[dict] = []
for w in words:
    data = search_word(w)
    all_items.extend(data["channel"]["item"])
    time.sleep(0.3)

pos_counter: Counter = Counter(
    item["sense"][0].get("pos") or "(미상)" for item in all_items
) # pos는 sense 안에 있음

top3: list[tuple[str, int]] = pos_counter.most_common(3)
for pos, count in top3:
    print(f"{pos}: {count}건")

명사: 60건
(미상): 19건
어미: 1건


(2) 코드 실행 결과 : 명사: 60건
(미상): 19건
어미: 1건

(3) 설명 : 8개 검색어에서 받은 항목들을 all_items에 합친 뒤, 각 항목의 품사를 꺼내 collections.Counter에 넣고 most_common(3)으로 가장 많이 나온 품사 3개와 그 빈도를 출력했다. 처음에는 힌트대로 item.get("pos")를 썼더니 전부 미상으로 처리되어 json.dumps로 한 항목의 구조를 출력해 확인했고, 그 결과 pos가 sense 리스트의 0번 인덱스 안에 있음을 보고 코드를 수정했다. 

관찰 : 가장 흔한 품사는 명사이다. 그 까닭은 8개 검색어가 모두 명사인 음식 이름이라 표제어와 관련 어휘가 대부분 명사로 등록되어 있고, 한국어 사전 표제어 자체도 명사 비중이 가장 크기 때문에 명사가 가장 많이 집계된 것으로 보인다.